# Modelos y entrenamiento

En este notebook usamos el dataset ya transformado en la parte anterior. El objetivo es comparar dos modelos diferentes para predecir si una persona gana más o menos de 50K.

Modelos utilizados:

- Random Forest
- LightGBM

Para cada modelo entrenamos una version baseline y otra con ajuste de hiperparametros usando RandomizedSearchCV y validación cruzada.

## 1. Carga de librerias y datos

In [ ]:
import warnings

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

#Ayuda de la IA para meter el filter warnings porque me imprimia muchas lineas de warnings y asi las ignoro y no salen.

In [ ]:
X = pd.read_csv('data/X_processed.csv')
y = pd.read_csv('data/y_processed.csv').squeeze()

print('Dimensiones X:', X.shape)
print('Dimensiones y:', y.shape)
print('\nDistribucion del target:')
print(y.value_counts(normalize=True).round(3))

La clase 0 representa ingresos menores o iguales a 50k y la clase 1 representa ingresos superiores a 50K. Podemos ver que hay mas casos de la clase 0, por lo que no conviene mirar solo el accuracy.

## 2. División en train y test

Usamos el 80% de los datos para entrenar y el 20% para probar. La opción stratify=y mantiene una proporcion parecida de clases en train y test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)

## 3. Funcion para calcular métricas

Aqui calculamos las métricas principales que hemos visto en clase: accuracy, precision, recall, F1 y ROC AUC. También revisamos la matriz de confusión.

In [ ]:
#Ayuda de la IA para crear una función sencilla para no repetir el mismo cálculo de metricas varias veces.
def evaluar_modelo(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    return {
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }

## 4. Modelo 1: Random Forest

Primero entrenamos un Random Forest baseline sin ajuste de hiperparametros. Después una búsqueda aleatoria de parametros con validacion cruzada.

In [ ]:
rf_baseline = RandomForestClassifier(random_state=42, n_jobs=1)
rf_baseline.fit(X_train, y_train)

rf_baseline_resultado = evaluar_modelo('Random Forest baseline', rf_baseline, X_test, y_test)
rf_baseline_resultado

In [ ]:
#Un poco de ayuda de la IA para crear el codigo bien y no fallar al buscar

rf_parametros = {
    'n_estimators': [80, 120, 160],
    'max_depth': [8, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=1),
    param_distributions=rf_parametros,
    n_iter=6,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=1
)

rf_search.fit(X_train, y_train)

print('Mejores parametros Random Forest:')
print(rf_search.best_params_)
print('Mejor F1 medio en validacion cruzada:', round(rf_search.best_score_, 4))

In [ ]:
rf_tuned = rf_search.best_estimator_
rf_tuned_resultado = evaluar_modelo('Random Forest fine tuning', rf_tuned, X_test, y_test)
rf_tuned_resultado

## 5. Modelo 2: LightGBM

Aqui usamos un LightGBM también basado en arboles, pero este funciona mediante boosting. Lo entrenamos primero con una version baseline y luego otra con fine tuning.

In [ ]:
lgbm_baseline = LGBMClassifier(random_state=42, verbose=-1)
lgbm_baseline.fit(X_train, y_train)

lgbm_baseline_resultado = evaluar_modelo('LightGBM baseline', lgbm_baseline, X_test, y_test)
lgbm_baseline_resultado

In [ ]:
#Igual que con el modelo de RandomForest, ayuda sutil.

lgbm_parametros = {
    'n_estimators': [80, 120, 160],
    'learning_rate': [0.03, 0.05, 0.1],
    'num_leaves': [15, 31, 63],
    'max_depth': [-1, 6, 10],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

lgbm_search = RandomizedSearchCV(
    estimator=LGBMClassifier(random_state=42, verbose=-1),
    param_distributions=lgbm_parametros,
    n_iter=6,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=1
)

lgbm_search.fit(X_train, y_train)

print('Mejores parametros LightGBM:')
print(lgbm_search.best_params_)
print('Mejor F1 medio en validacion cruzada:', round(lgbm_search.best_score_, 4))

In [ ]:
lgbm_tuned = lgbm_search.best_estimator_
lgbm_tuned_resultado = evaluar_modelo('LightGBM fine tuning', lgbm_tuned, X_test, y_test)
lgbm_tuned_resultado

## 6. Comparación de resultados

In [ ]:
resultados = pd.DataFrame([
    rf_baseline_resultado,
    rf_tuned_resultado,
    lgbm_baseline_resultado,
    lgbm_tuned_resultado
]).set_index('modelo')

resultados.round(4)
#Con .round(4) redondeo a 4 decimales.

In [ ]:
resultados[['accuracy', 'f1', 'roc_auc']].plot(kind='bar', figsize=(10, 5))
plt.title('Comparacion de modelos')
plt.ylabel('Puntuacion')
plt.ylim(0, 1)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 7. Matrices de confusión

Para no hacer demasiadas gráficas, aqui mostramos las matrices de confusión de las versiones con fine tuning de cada modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, nombre, modelo in [
    (axes[0], 'Random Forest tuning', rf_tuned),
    (axes[1], 'LightGBM tuning', lgbm_tuned)
]:
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(nombre)

plt.tight_layout()
plt.show()

In [ ]:
print('Random Forest fine tuning')
print(classification_report(y_test, rf_tuned.predict(X_test)))

print('LightGBM fine tuning')
print(classification_report(y_test, lgbm_tuned.predict(X_test)))

## 8. Importancia de variables

Revisamos las variables más importantes del mejor modelo para interpretar un poco el resultado.

In [ ]:
mejor_modelo = lgbm_tuned

importancias = pd.Series(
    mejor_modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(10)

importancias.sort_values().plot(kind='barh', figsize=(8, 5))
plt.title('Variables mas importantes en LightGBM')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

importancias

#Con importancias muestro directamente el contenido de esa variable en el notebook

## 9. Conclusiones

- Hemos probado dos modelos diferentes, Random Forest y LightGBM.

- Para cada modelo hemos hecho una versión baseline y otra con fine tuning usando RandomizedSearchCV con validación cruzada de 3 particiones.

- El dataset esta desbalanceado: aproximadamente el 75% de los registros son <=50K y el 25% son >50K. Por eso se hemos revisado también precision, recall, F1 y ROC AUC.

- Random Forest mejora un poco con el ajuste de hiperparametros. El F1 pasa aproximadamente de 0.663 a 0.673.

- LightGBM obtiene los mejores resultados generales. Su versión ajustada consigue un accuracy de unos 0.871, un F1 de 0.712 y un ROC AUC de 0.928.

- La diferencia entre LightGBM baseline y LightGBM con tuning es pequeña, por lo que el modelo ya funcionaba bastante bien con los parámetros por defecto.

- La clase <=50K se predice mejor que la clase >50K, principalmente porque hay mas ejemplos de esa clase en el dataset.

- Las variables más relevantes estan relacionadas con educación, ganancias de capital, edad, horas trabajadas y situacion familiar/laboral. Esto tiene sentido porque son factores que pueden influir en el nivel de ingresos.

- Como conclusión final, el mejor modelo elegido seria LightGBM con fine tuning, aunque Random Forest también ofrece resultados buenos y es más fácil de interpretar.

#No he metido formato codigo en ningún markdown para evitar confusión.